# Jour 3 — Autoencodeur et erreur de reconstruction

> **Version corrigée** — les cellules de solution sont signalées.


## Objectifs

- comprendre encodeur, espace latent et décodeur
- entraîner un réseau dont la cible est l’entrée
- calculer une erreur de reconstruction par image
- transformer cette erreur en signal d’observation inhabituelle


## Apprendre sans label de classe

![Architecture d’un autoencodeur](../../ressources/illustrations/jour_03/03_architecture_autoencodeur.png)

Un **autoencodeur** reçoit une donnée et apprend à la reconstruire. La cible est donc l’entrée elle-même.

L’**encodeur** transforme les pixels en une représentation plus compacte. Le **décodeur** tente de reconstruire l’image à partir de cette représentation. La zone compacte entre les deux s’appelle l’**espace latent** ou goulot d’étranglement.

Ce n’est pas une copie directe : limiter la taille latente oblige le réseau à retenir certaines régularités communes aux données d’entraînement.


## L’espace latent

![Exemple de projection d’un espace latent](../../ressources/illustrations/jour_03/03_espace_latent.png)

Chaque image devient un vecteur latent. Deux vecteurs proches indiquent que l’encodeur produit des représentations similaires selon ce qu’il a appris.

L’espace latent peut contenir huit, trente-deux ou davantage de dimensions. Une visualisation en deux dimensions est seulement une projection simplifiée.

Les groupes visibles ne correspondent pas automatiquement aux catégories humaines. L’autoencodeur optimise la reconstruction, pas la séparation des classes.


## De la reconstruction à l’observation inhabituelle

![Erreur de reconstruction faible ou élevée](../../ressources/illustrations/jour_03/03_reconstruction_anomalie.png)

Si le réseau est entraîné uniquement sur des exemples habituels, il peut reconstruire moins fidèlement une forme différente. L’**erreur de reconstruction** mesure l’écart pixel par pixel entre entrée et sortie.

Une erreur élevée devient un signal d’alerte possible, mais pas une preuve. Certaines observations inhabituelles restent faciles à reconstruire et certains exemples normaux difficiles peuvent produire une grande erreur.

Le seuil doit donc être étudié sur des données séparées et relié au coût des faux positifs et faux négatifs.


![Distributions d’erreur et seuil de reconstruction](../../ressources/illustrations/jour_03/03_seuil_reconstruction.png)

### Pourquoi un seuil parfait est rare

Les erreurs des cas habituels et différents peuvent se chevaucher. Un exemple habituel difficile peut dépasser le seuil : c’est un faux positif. Un cas différent facile à reconstruire peut rester dessous : il sera manqué.

Le 95e percentile n’est pas une constante universelle. Il fournit ici une règle simple : seules 5 % environ des erreurs du train habituel le dépassent. Une autre application pourrait choisir 90 %, 99 % ou un seuil défini à partir des coûts métier.

Le seuil doit être calculé sans regarder les données utilisées pour l’évaluation finale. Sinon, on adapte indirectement la décision aux réponses que l’on prétend mesurer.


## Une expérience contrôlée avec les chiffres

Nous entraînons l’autoencodeur uniquement sur les chiffres de 0 à 4. Ils jouent ici le rôle de la famille habituelle. Les chiffres de 5 à 9 servent de famille différente.

Cette séparation est pédagogique. Elle ne signifie pas qu’un chiffre 8 est objectivement une anomalie : il est seulement absent des exemples fournis à l’entraînement.

Les images 8 × 8 sont aplaties en 64 valeurs. La sortie utilise `sigmoid` car les pixels normalisés restent entre 0 et 1.


`reshape(len(images), -1)` garde une ligne par image et déduit les 64 pixels. Les conditions `labels <= 4` et `labels >= 5` sélectionnent les deux familles avec des masques booléens.

La syntaxe Keras `Dense(...)(inputs)` signifie « créer cette couche, puis la relier aux sorties nommées inputs ». Chaque nom conserve une étape du calcul. `Model(inputs, outputs)` décrit le chemin complet ; `Model(inputs, latent)` décrit seulement l'encodeur, avec les mêmes poids.

Dans `fit(usual_train, usual_train)`, l'entrée et la réponse sont la même image : le réseau apprend à reconstruire. La MSE mesure les écarts des pixels. Sigmoid garde les pixels reconstruits entre 0 et 1.


In [ ]:
# Réglages de l’environnement, avant le chargement de TensorFlow.
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

# NumPy gère les tableaux et les calculs numériques.
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# Fixe le hasard pour faciliter les comparaisons.
tf.keras.utils.set_random_seed(42)
try:
    # Limite les ressources utilisées par cette petite démonstration.
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass

# Charge les images et leurs labels ensemble.
digits = load_digits()
# Convertit les pixels en décimaux et les ramène à l’échelle [0, 1].
images = digits.images.astype("float32") / 16.0


In [ ]:
# Conserve un chiffre entier comme bonne réponse par image.
labels = digits.target.astype("int64")
# Réorganise les pixels sans changer leurs valeurs.
flat_images = images.reshape(len(images), -1)

# Sélectionne les chiffres habituels grâce aux labels.
usual = flat_images[labels <= 4]
# Sélectionne la famille absente de l’entraînement.
unfamiliar = flat_images[labels >= 5]
# Sépare ensemble les entrées et les réponses pour garder leur association.
usual_train, usual_test = train_test_split(
    usual, test_size=0.25, random_state=42
)

# Décrit les 64 valeurs fournies à l’encodeur.
inputs = tf.keras.layers.Input(shape=(64,))
# Relie une couche Dense aux entrées décrites.
hidden = tf.keras.layers.Dense(32, activation="relu")(inputs)
# Réduit l’image à huit valeurs apprises.
latent = tf.keras.layers.Dense(8, activation="relu", name="latent")(hidden)
# Commence à reconstruire les pixels à partir du résumé.
hidden_decoded = tf.keras.layers.Dense(32, activation="relu")(latent)
# Produit les 64 pixels reconstruits, bornés entre 0 et 1.
outputs = tf.keras.layers.Dense(64, activation="sigmoid")(hidden_decoded)


In [ ]:
# Relie l’entrée à la reconstruction complète.
autoencoder = tf.keras.Model(inputs, outputs, name="autoencodeur")
# Expose seulement le résumé appris, avec les mêmes couches.
encoder = tf.keras.Model(inputs, latent, name="encodeur")
# Associe le réseau, sa loss et l’optimiseur ; aucun poids ne change encore.
autoencoder.compile(optimizer="adam", loss="mse")

# Suit la validation et restaure les meilleurs poids observés.
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True
)
# Entraîne les poids sur le train et observe la validation séparément.
history = autoencoder.fit(
    usual_train, usual_train,
    validation_split=0.20,
    epochs=40,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)
print("Époques effectuées :", len(history.history["loss"]))
print("Dimension d'entrée :", usual_train.shape[1])
print("Dimension latente :", encoder.output_shape[1])


## Regarder avant de résumer

Une moyenne peut masquer des reconstructions très différentes. Nous affichons donc quelques images habituelles et quelques images de la famille non vue.

Une reconstruction plus floue n’est pas forcément inutile. Le réseau cherche à reproduire les grandes régularités avec seulement huit valeurs latentes.


`reshape(8, 8)` remet une ligne de 64 valeurs en grille pour l'afficher. Cela ne modifie ni les valeurs ni les poids. Chaque colonne du dessin correspond à un exemple ; les lignes comparent original habituel, reconstruction, original différent et reconstruction.

Les mêmes limites de gris, de 0 à 1, sont utilisées partout pour que le contraste soit comparable.


In [ ]:
# Reconstruit quelques images habituelles restées hors du train.
usual_reconstructed = autoencoder.predict(usual_test[:4], verbose=0)
# Teste aussi des chiffres jamais utilisés pour apprendre.
unfamiliar_reconstructed = autoencoder.predict(unfamiliar[:4], verbose=0)

# Réserve les emplacements graphiques, sans modifier les données.
fig, axes = plt.subplots(4, 4, figsize=(7, 7))
for i in range(4):
    axes[0, i].imshow(usual_test[i].reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
    axes[1, i].imshow(usual_reconstructed[i].reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
    axes[2, i].imshow(unfamiliar[i].reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
    axes[3, i].imshow(unfamiliar_reconstructed[i].reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
    for row in range(4):
        axes[row, i].axis("off")
axes[0, 0].set_ylabel("habituel", rotation=0, labelpad=32)
axes[1, 0].set_ylabel("reconstruit", rotation=0, labelpad=32)
axes[2, 0].set_ylabel("différent", rotation=0, labelpad=32)
axes[3, 0].set_ylabel("reconstruit", rotation=0, labelpad=32)
plt.tight_layout()
# Affiche le résultat des calculs précédents.
plt.show()


## Une erreur par image

La MSE utilisée pendant l’entraînement moyenne toutes les erreurs du batch. Pour étudier chaque observation, nous calculons une MSE séparée par image.

Deux distributions qui se chevauchent signifient qu’aucun seuil ne séparera parfaitement les familles. Le seuil de 95e percentile du train, utilisé dans l’exercice, accepte environ 95 % des erreurs observées sur le train et signale les 5 % les plus élevées.


`(values - reconstructed) ** 2` calcule l'écart au carré de chaque pixel. `mean(axis=1)` moyenne les 64 écarts de chaque image et produit une erreur par image.

Le graphique en boîtes résume les distributions : la ligne centrale montre la médiane, c'est-à-dire la valeur du milieu ; la boîte montre la moitié centrale des valeurs. `showfliers=False` masque les points isolés du dessin sans les supprimer des calculs. La médiane n'est pas la moyenne.


In [ ]:
# Fonction réutilisable : calcule une erreur moyenne par image.
def reconstruction_error(model, values):
    reconstructed = model.predict(values, verbose=0)
    # Moyenne les erreurs au carré sur les pixels de chaque image.
    return np.mean((values - reconstructed) ** 2, axis=1)

# Erreurs de référence, calculées sans consulter les exemples d’évaluation.
train_errors = reconstruction_error(autoencoder, usual_train)
# Mesure les images habituelles restées hors de l’apprentissage.
usual_errors = reconstruction_error(autoencoder, usual_test)
# Mesure les images de la famille différente.
unfamiliar_errors = reconstruction_error(autoencoder, unfamiliar)

plt.figure(figsize=(7, 3.5))
plt.boxplot(
    [usual_errors, unfamiliar_errors],
    tick_labels=["chiffres 0–4", "chiffres 5–9"],
    showfliers=False,
)
plt.ylabel("MSE de reconstruction")
plt.title("Les distributions peuvent se chevaucher")
# Affiche le résultat des calculs précédents.
plt.show()

print("Médiane habituelle :", round(float(np.median(usual_errors)), 4))


In [ ]:
# Lecture des résultats : aucune modification des poids dans cette cellule.
print("Médiane différente :", round(float(np.median(unfamiliar_errors)), 4))


## À vous de jouer — choisir et évaluer un seuil

Fixez `threshold` au 95e percentile de `train_errors`.

Calculez ensuite :

- le taux de chiffres habituels signalés à tort ;
- le taux de chiffres différents détectés ;
- le nombre d’observations de chaque groupe.

Interprétez les deux taux ensemble : augmenter le seuil réduit généralement les fausses alertes, mais peut aussi manquer davantage de cas différents.

`np.percentile(train_errors, 95)` donne une valeur telle qu’environ 95 % des erreurs du train sont en dessous. La moyenne d’une comparaison booléenne compte True comme 1 et False comme 0, pour obtenir une proportion.


In [ ]:
# Correction
threshold = float(np.percentile(train_errors, 95))
false_positive_rate = float(np.mean(usual_errors > threshold))
detection_rate = float(np.mean(unfamiliar_errors > threshold))

print("Seuil :", round(threshold, 4))
print("Cas habituels évalués :", len(usual_errors))
print("Cas différents évalués :", len(unfamiliar_errors))
print("Taux de faux positifs :", f"{false_positive_rate:.1%}")
print("Taux de détection des cas différents :", f"{detection_rate:.1%}")


## Limites à retenir

Un autoencodeur très puissant peut apprendre à reconstruire presque tout, y compris des observations différentes. Un goulot trop petit peut au contraire mal reconstruire de nombreux cas habituels.

La qualité dépend de la représentativité du train, de l’architecture, de la loss et du seuil. Une erreur élevée doit déclencher une analyse, pas une conclusion automatique sur la nature du cas.
